In [1]:
import time
import json
import os
import numpy as np
import matplotlib.pyplot as plt

import os
import sys

sys.path.append(os.path.dirname(os.getcwd()))
from pc import PS
from modules import ADC,DAC,CHIP,SELECT
from command import CMD,CmdData,Packet
from command.singleCmdInfo import *

from util import plot_v_cond,plot_cond,show_crossbar,DataLoader
import pickle
from network.layer import Layer,hnnLayer

In [2]:
chip=CHIP(PS(host="192.168.1.12", port = 7, debug=0),init=True)
chip.set_device_cfg(deviceType=0,IsNew32=True)
chip.adc.set_gap(adc_cs_gap=100,adc_first_gap=600,adc_last_gap=20)
chip.adc.set_gain_resistor(big_resistance=22e-3,small_resistance=200)
chip.adc.set_sample_times(adc_sample_times=4)
chip.clk_manager.set_cyc(delay1=10,delay2=10,delay3=50)
chip.add_compiler("./compiler/code/")

Failed to connect: [WinError 10049] 在其上下文中，该请求的地址无效。
Failed to send message:
Failed to send message:
Failed to send message:
Failed to send message:
Failed to send message:
Failed to send message:
Failed to send message:
Failed to send message:
Failed to send message:
Failed to send message:
Failed to send message:
正在编译文件:  ./compiler/code/asm_ds(2).txt
正在编译文件:  ./compiler/code/汇编指令解释测试.txt
编译指令 jmp_r reg2 时出错: 'COMPILER' object has no attribute 'jmp_r'


In [3]:
print(chip.setting._numToBank_Index(128))

(2, 0)


In [26]:
ps_ddr_pos_start = 0xA100000 // 0x20  # 至少从这个ddr地址之后存放
ps_ddr_pos = ps_ddr_pos_start

In [27]:
ins_data = [
    CMD(PL_SET_MODE,command_data=CmdData(0)), # 2是神经元，0是推理，
]
chip.execute_ins(ins_data=ins_data,ins_ram_start=0)

完整字节码: 55aa04000000023c0000000b000000
模式: 4
	帧头:               	字节码: 55 aa 04
	指令: pl_ram_addr   	字节码: 0000
	指令: pl_data_length	字节码: 0002
	指令: pl_set_mode   	字节码: 3c000000
	指令: pl_exit       	字节码: 0b000000

Failed to send message:
完整字节码: 55aa010100008000
模式: 1
	帧头:               	字节码: 55 aa 01
	指令: fast_command_1	字节码: 0100008000

Failed to send message:


### 这个是准备神经元的映射表

In [28]:
ins_data = []
for i in range(128):
    bank,index = chip.setting.get_bank_index32([i+128])
    ins_data.append(CMD(PL_DATA,command_data=CmdData(i<<16 | bank)))
    ins_data.append(CMD(PL_DATA,command_data=CmdData(index)))

ins_data.append(CMD(PL_INIT_NEURON_MAP,command_data=CmdData(0)))  # 初始化神经元映射表，传到dout_ram地址8开始

chip.execute_ins(ins_data=ins_data,ins_ram_start=0)

完整字节码: 55aa04000001020000000400000001000100400000000100020004000000020003004000000002000400040000000400050040000000040006000400000008000700400000000800080004000000100009004000000010000a000400000020000b004000000020000c000400000040000d004000000040000e000400000080000f0040000000800010000400000100001100400000010000120004000002000013004000000200001400040000040000150040000004000016000400000800001700400000080000180004000010000019004000001000001a000400002000001b004000002000001c000400004000001d004000004000001e000400008000001f0040000080000020000400010000002100400001000000220004000200000023004000020000002400040004000000250040000400000026000400080000002700400008000000280004001000000029004000100000002a000400200000002b004000200000002c000400400000002d004000400000002e000400800000002f0040008000000030000401000000003100400100000000320004020000000033004002000000003400040400000000350040040000000036000408000000003700400800000000380004100000000039004010000000003a000420000000003b004020000000003c000440000000003

In [ ]:
# 配置电压为0.1v
ins_data = chip.get_dac_ins2(v=0.1)
din_ram_start = 0
for image in range(10):
    row_index_input = [i for i in range(128)]
    col_index_input = [i for i in range(256)]
    for
    pre_ins_data,din_ram_data,operator_batch,res_tia_map = chip.prepare_latch_ins4(None,row_index=row_index_input,col_index=col_index_input,din_ram_start=din_ram_start,from_row=False,split_type=4)

    # 发送din_ram到DDR
    ps_ddr_pos = chip.send_ps_ddr5(din_ram_data,mode=8,ps_ddr_pos=ps_ddr_pos)

    for i,ins in enumerate(pre_ins_data):
        ins.append(CMD(PL_START_NEURON_ACCU,command_data=CmdData(int(i==0)<<1 | 0))) # 启动神经元累加
        ins.append(CMD(PL_SET_NEURON_BANK_INDEX,command_data=CmdData(0xFF<<8 | din_ram_start))) # 设置神经元的bank和index，先清零
        ins.append(CMD(PL_SET_NEURON_BANK_INDEX,command_data=CmdData(1<<(8+i) | din_ram_start+1))) # 设置神经元的bank和index，再配1
        ins.append(CMD(PL_READ_COL_PULSE,command_data=CmdData(i))) # 读取列脉冲计数，dout_ram固定0~7行（每行512bit）存储神经元的输出脉冲，映射关系从8行开始存放
        
        # 未超过界限，就继续往指令包里面加指令，否则先发一轮
        # if len(ins_data)+len(ins)+1 >= chip.setting.ins_ram_length:
        #     ps_ddr_pos = chip.send_ps_ddr5(ins_data,mode=9,ps_ddr_pos=ps_ddr_pos)

        # ins_data.extend(ins)
        # ins_data.append(CMD(PL_EXIT))

        ins_data.extend(ins)
        ins_data.append(CMD(PL_EXIT))
        ps_ddr_pos = chip.send_ps_ddr5(ins_data,mode=9,ps_ddr_pos=ps_ddr_pos)
        ps_ddr_pos = chip.send_ps_ddr5(ddr_data=chip.adc.get_out_ins5(data_length=8,dout_ram_start=0),mode=12,ps_ddr_pos=ps_ddr_pos)

完整字节码: 55aa08005080660000000a00000000ffffffff8080808040404040202020201010101008080808040404040202020201010101
模式: 8
	帧头:               	字节码: 55 aa 08
	指令: ps_ddr_addr   	字节码: 00508066
	指令: ps_data_length	字节码: 0000000a
	指令: pl_data       	字节码: 00000000
	指令: pl_data       	字节码: ffffffff
	指令: pl_data       	字节码: 80808080
	指令: pl_data       	字节码: 40404040
	指令: pl_data       	字节码: 20202020
	指令: pl_data       	字节码: 10101010
	指令: pl_data       	字节码: 08080808
	指令: pl_data       	字节码: 04040404
	指令: pl_data       	字节码: 02020202
	指令: pl_data       	字节码: 01010101

Failed to send message:
完整字节码: 55aa08005080690000000a00000000ffffffff8080808040404040202020201010101008080808040404040202020201010101
模式: 8
	帧头:               	字节码: 55 aa 08
	指令: ps_ddr_addr   	字节码: 00508069
	指令: ps_data_length	字节码: 0000000a
	指令: pl_data       	字节码: 00000000
	指令: pl_data       	字节码: ffffffff
	指令: pl_data       	字节码: 80808080
	指令: pl_data       	字节码: 40404040
	指令: pl_data       	字节码: 20202020
	指令: pl_data       	字节码: 1010

In [22]:
ps_ddr_pos = chip.send_ps_run(flag=2,ps_ddr_pos_start=ps_ddr_pos_start,ps_ddr_pos=ps_ddr_pos)

完整字节码: 55aa0b005080640100508000
模式: 11
	帧头:               	字节码: 55 aa 0b
	指令: ps_ddr_addr   	字节码: 00508064
	指令: ps_start_finsh	字节码: 01
	指令: ps_ddr_addr   	字节码: 00508000

Failed to send message:
完整字节码: 55aa0b005080650000508000
模式: 11
	帧头:               	字节码: 55 aa 0b
	指令: ps_ddr_addr   	字节码: 00508065
	指令: ps_start_finsh	字节码: 00
	指令: ps_ddr_addr   	字节码: 00508000

Failed to send message:
